# 🔹 Notebook 3 — Pipeline RAG con Knowledge Graph

**Obiettivo**: Testare il flusso completo RAG — dalla domanda utente alla risposta grounded.

Combina: **Vector Search** (Redis) + **Graph Traversal** (Neo4j) + **LLM** (Ollama)


## Setup: connessioni

In [ ]:
import os, json, struct
import httpx
import redis
from neo4j import GraphDatabase

OLLAMA_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
LLM_MODEL = os.getenv('OLLAMA_LLM_MODEL', 'qwen2.5:14b')
EMB_MODEL = os.getenv('OLLAMA_EMBEDDING_MODEL', 'nomic-embed-text')
REDIS_URL = os.getenv('REDIS_URL', 'redis://localhost:6379')
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_AUTH = (os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD', 'yourpassword'))

r = redis.from_url(REDIS_URL, decode_responses=True)
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)

def get_embedding(text):
    resp = httpx.post(f'{OLLAMA_URL}/api/embed', json={'model': EMB_MODEL, 'input': text}, timeout=60)
    return resp.json()['embeddings'][0]

def chat(prompt, system='Sei un assistente aziendale. Rispondi SOLO con i fatti forniti nel contesto.'):
    resp = httpx.post(f'{OLLAMA_URL}/api/chat', json={
        'model': LLM_MODEL, 'stream': False,
        'messages': [{'role':'system','content':system}, {'role':'user','content':prompt}]
    }, timeout=120)
    return resp.json()['message']['content']

print('Tutti i servizi connessi.')

## 3.1 — RAG Step 1: Vector Search

In [ ]:
from redis.commands.search.query import Query

question = 'Qual e la garanzia per il prodotto SlimWash Pro?'
print(f'Domanda utente: "{question}"\n')

# Step 1: embedding della domanda
q_emb = get_embedding(question)
q_bytes = struct.pack(f'{len(q_emb)}f', *q_emb)

# Step 2: ricerca semantica su Redis
q = (Query('*=>[KNN 3 @embedding $vec AS score]')
     .sort_by('score').return_fields('text','doc_name','score').dialect(2))
vector_results = r.ft('idx:kg_chunks').search(q, query_params={'vec': q_bytes})

print('Chunk trovati via Vector Search:')
vector_context = []
for doc in vector_results.docs:
    vector_context.append(doc.text)
    print(f'  [{float(doc.score):.3f}] {doc.text[:100]}...')

## 3.2 — RAG Step 2: Graph Traversal

In [ ]:
# Step 3: query strutturata sul grafo per entita' menzionate
with neo4j_driver.session() as session:
    graph_results = session.run("""
        MATCH (p:Product {name: 'SlimWash Pro'})
              -[:HAS_WARRANTY]->(w:WarrantyPolicy)
        OPTIONAL MATCH (p)-[:COMPATIBLE_WITH]->(compat:Product)
        RETURN p.name AS prodotto, p.price AS prezzo,
               w.duration AS garanzia, w.coverage AS copertura,
               collect(compat.name) AS compatibili
    """)
    graph_context = [record.data() for record in graph_results]

print('Fatti dal Knowledge Graph:')
for g in graph_context:
    print(f'  Prodotto: {g["prodotto"]} (EUR {g["prezzo"]})')
    print(f'  Garanzia: {g["garanzia"]} — {g["copertura"]}')
    print(f'  Compatibili: {g["compatibili"]}')

## 3.3 — RAG Step 3: LLM Grounded Generation

In [ ]:
# Step 4: assembla contesto e genera risposta
context_parts = []
context_parts.append('--- FATTI DAL KNOWLEDGE GRAPH ---')
for g in graph_context:
    context_parts.append(f'Prodotto: {g["prodotto"]}, Prezzo: EUR {g["prezzo"]}')
    context_parts.append(f'Garanzia: {g["garanzia"]}, Copertura: {g["copertura"]}')
    context_parts.append(f'Prodotti compatibili: {", ".join(g["compatibili"])}')

context_parts.append('\n--- CHUNK DI TESTO RILEVANTI ---')
for chunk in vector_context[:2]:
    context_parts.append(chunk)

full_context = '\n'.join(context_parts)

prompt = f"""Contesto verificato:\n{full_context}\n\nDomanda: {question}\n\nRispondi SOLO con i fatti del contesto. Se non trovi la risposta, dillo."""

print('Risposta LLM (grounded):\n')
answer = chat(prompt)
print(answer)

## 3.4 — Confronto: LLM senza contesto vs con KG

In [ ]:
# Senza contesto (rischio hallucination)
print('=== SENZA Knowledge Graph ===')
naive_answer = chat(question, system='Sei un assistente. Rispondi alla domanda.')
print(naive_answer)
print()
print('=== CON Knowledge Graph (RAG) ===')
print(answer)
print()
print('Nota: la risposta con KG e verificabile e tracciabile.')

## Cleanup

In [ ]:
neo4j_driver.close()
print('Connessioni chiuse.')